# Ejercicios 07. Correlación

Antes de empezar:

1. Repasa el notebook de teoría `notebooks/07_correlacion.ipynb`.
2. Ejecuta la celda de preparación, que es la primera celda de código.
3. Busca las marcas `COMPLETA` y cambia los `...` por tu código.
4. Ejecuta las celdas de verificación. Imprimen "Correcto" si tu resultado coincide y "Revisa" si no.
5. Contesta las preguntas de reflexión con tus propias palabras. Es la parte más importante.

Si una celda marca error, lee el último renglón del mensaje; casi siempre ahí está la explicación.
Por ejemplo, `NameError` quiere decir que usaste una variable que no existe o que no ejecutaste
una celda anterior.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## Ejercicio 1. Correlación a mano

Para $x = \{1, 2, 3\}$, $y = \{1, 0, -1\}$ (ambas desde $n=0$):
a) ¿Qué lags tiene $r_{xy}$? b) Calcula $r_{xy}[l]$ a mano. c) Compruébalo con `pds.correlacion_cruzada`.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([1.0, 0.0, -1.0])

lags_esperados = ...               # COMPLETA (np.arange)
r_mano = np.array([...])           # COMPLETA a mano

lags, r = ...                      # COMPLETA
print(lags, r)

In [ ]:
pds.comprobar_iguales(lags, lags_esperados, "lags de -2 a 2")
pds.comprobar_iguales(r, r_mano, "valores de la correlación")

## Ejercicio 2. La correlación NO es conmutativa

Con las mismas $x$, $y$: calcula $r_{yx}$ y compárala con $r_{xy}$. ¿Qué relación tienen? Compruébala.

In [ ]:
lags_yx, r_yx = ...                  # COMPLETA
print("r_xy =", r)
print("r_yx =", r_yx)
relacion = ...                        # COMPLETA: escribe r_yx en términos de r (pista: [::-1])

In [ ]:
pds.comprobar_iguales(r_yx, relacion, "r_yx[l] = r_xy[-l]")

## Ejercicio 3. Sonar: ¿a qué distancia está el objeto?

Un sonar submarino trabaja con $f_s = 20$ kHz. El sonido en el agua viaja a $c = 1500$ m/s.
Como pulso usa una **secuencia pseudoaleatoria** de 1000 muestras (50 ms) (valores ±1), más fácil de detectar que una senoidal.
El eco regresa con amplitud 0.2, escondido en ruido de amplitud 1.

a) Simula la señal recibida (0.5 s) con un objeto a **150 m** (calcula el retardo $D$ en muestras).
b) Estima el retardo con `pds.estimar_retardo` y calcula la distancia.
c) Grafica la señal recibida y la correlación.

In [ ]:
fs = 20000
c = 1500
generador = np.random.default_rng(7)
pulso = np.sign(generador.standard_normal(1000))         # ±1 aleatorios

distancia_real = 150
tiempo_ida_vuelta = ...                                  # COMPLETA [s]
D_real = ...                                             # COMPLETA [muestras, entero]

recibida = 1.0 * generador.standard_normal(pds.indice_de_muestra(0.5, fs))
recibida[D_real:D_real + len(pulso)] += 0.2 * pulso

D_estimado = ...                                         # COMPLETA
distancia_estimada = ...                                 # COMPLETA
print(f"D real = {D_real}, D estimado = {D_estimado}, distancia = {distancia_estimada:.2f} m")

lags, r = pds.correlacion_cruzada(recibida, pulso)
figura, ejes = plt.subplots(2, 1, figsize=(10, 6))
pds.graficar_continua(np.arange(len(recibida)) / fs, recibida, titulo="Señal recibida", ax=ejes[0])
pds.graficar_continua(lags / fs * c / 2, r, titulo="Correlación", etiqueta_x="distancia (m)", ax=ejes[1])
plt.tight_layout()
plt.show()

In [ ]:
pds.comprobar(D_real == 4000, "D real = 4000 muestras (0.2 s)")
pds.comprobar(D_estimado == D_real, "el retardo se estima exactamente")
pds.comprobar_iguales(distancia_estimada, 150, "distancia = 150 m")

**Reflexión:** la amplitud del eco (0.2) es 5 veces **menor** que la del ruido. ¿Por qué aun así la correlación lo encuentra? ¿Qué pasaría con un pulso más corto (50 muestras)? Pruébalo.

## Ejercicio 4. Periodo de la señal misteriosa con ruido

`../datos/senal_misteriosa_ruido.csv` es la señal misteriosa del notebook 04 (periodo 12) con mucho ruido.
a) Comprueba que `pds.estimar_periodo` **falla**. b) Encuentra el periodo con la autocorrelación.

In [ ]:
n_m, x_m = pds.cargar_senal_texto("../datos/senal_misteriosa_ruido.csv", fs=1)
print("Búsqueda directa:", pds.estimar_periodo(x_m))

lags, r = ...                          # COMPLETA
seleccion = ...                        # COMPLETA: lags >= 3
periodo = ...                          # COMPLETA
print("Periodo por autocorrelación:", periodo)

pds.graficar_discreta(lags[(lags >= -60) & (lags <= 60)], r[(lags >= -60) & (lags <= 60)],
                      titulo="Autocorrelación (lags -60 a 60)", etiqueta_x="l")
plt.show()

In [ ]:
pds.comprobar(periodo == 12, "periodo = 12")

**Reflexión:** en la gráfica también hay picos en $l = 4$ y $l = 8$ (aunque más bajos que en 12). ¿Por qué? (Pista: ¿de qué está hecha la señal?)

## Ejercicio 5. Frecuencia cardíaca con autocorrelación

Repite el cálculo de la frecuencia cardíaca del ejercicio 4 del notebook 03, ahora **sin detectar picos**:
la autocorrelación del ECG tiene un pico en el periodo del latido. Busca el máximo sólo entre **0.3 s y 1.5 s**
(de 40 a 200 latidos por minuto: los valores fisiológicamente posibles). Fuera de ese rango, la deriva de la línea
base (periodo ≈ 3.3 s) produce otros picos que no son el latido.

*Sugerencia:* resta primero la media del ECG (`ecg - np.mean(ecg)`) para que la deriva no domine.

In [ ]:
fs_ecg = 360
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)
ecg_centrado = ...                                     # COMPLETA

lags, r = ...                                          # COMPLETA
l_minimo = pds.indice_de_muestra(0.3, fs_ecg)
l_maximo = pds.indice_de_muestra(1.5, fs_ecg)
seleccion = ...                                        # COMPLETA: lags entre l_minimo y l_maximo (usa &)
periodo_muestras = ...                                 # COMPLETA
periodo_segundos = ...                                 # COMPLETA
lpm = ...                                              # COMPLETA
print(f"Periodo = {periodo_muestras} muestras = {periodo_segundos:.3f} s  ->  {lpm:.1f} lpm")

mostrar = (lags >= 0) & (lags <= 3 * fs_ecg)
pds.graficar_continua(lags[mostrar] / fs_ecg, r[mostrar], titulo="Autocorrelación del ECG", etiqueta_x="retardo (s)")
plt.show()

In [ ]:
pds.comprobar(abs(lpm - 75) < 1, "frecuencia cardíaca ≈ 75 lpm")

## Ejercicio 6. Encontrar un patrón dentro de una señal

En comunicaciones digitales, el receptor busca una **secuencia de sincronización** conocida para saber dónde empieza el mensaje.
a) Crea el patrón `patron = [1, 1, 1, -1, -1, 1, -1]` (código de Barker de 7). b) Escóndelo en la posición 300 de una señal
de 1000 muestras de ruido (amplitud 0.5). c) Encuentra la posición con la **correlación normalizada** y grafícala.

In [ ]:
patron = np.array([1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0])
generador = np.random.default_rng(3)
senal = 0.5 * generador.standard_normal(1000)
posicion_real = 300
# COMPLETA: suma el patrón a la señal a partir de posicion_real
...

lags, rho = ...                                    # COMPLETA: correlación normalizada de senal con patron
posicion_encontrada = ...                          # COMPLETA
print("Posición encontrada:", posicion_encontrada)

pds.graficar_continua(lags, rho, titulo="Correlación normalizada", etiqueta_x="l")
plt.show()

In [ ]:
pds.comprobar(posicion_encontrada == 300, "el patrón está en la posición 300")